# IMPORTS:

In [10]:
import warnings
warnings.filterwarnings('ignore')
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
import optuna
from optuna.pruners import MedianPruner
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import RFE
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

In [2]:
df = pd.read_csv("cardio_data_processed.csv")

# Preprocessing & Feature Engineering:

In [3]:
df.drop('id', axis=1, inplace = True) # Dropped id becoz its useless for prediction...
df["age"] = (df.age/365).astype(int) # Converting Age in years as its given in days...
df.drop('age_years', axis = 1, inplace = True) # Dropped id becoz its useless for prediction...
df.drop('bp_category_encoded', axis = 1, inplace = True) # Dropped id becoz its useless for prediction...
df.drop('bp_category', axis = 1, inplace = True) # Dropped id becoz its useless for prediction...

In [4]:
df["pulse_pressure"] = df["ap_hi"] - df["ap_lo"]
df["mean_arterial_pressure"] = df["ap_hi"] + 2*df["ap_lo"]/3

In [5]:
#Checking if there any missing or false value & dropping that...
df[['ap_hi', 'ap_lo', 'pulse_pressure', 'mean_arterial_pressure']].describe()
print((df['pulse_pressure'] < 0).sum()) #Since i got 3 value whose pulse_pressure is < 0 which make no sense as there is an error, so i gonna drop this 3 rows
df = df[df['pulse_pressure'] >= 0] #It removed those 3rows but index still show 0 to 68204 so we need to reset index...
df = df.reset_index(drop=True) #index reset...

3


# Train Test Split:

In [6]:
X = df.drop('cardio', axis = 1)
y = df['cardio']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 10)

# OPTUNA FOR RF:

In [8]:
def objective(trial):
    """
    Objective function for Optuna to optimize RF hyperparameters
    """
    
    # Define hyperparameter search space
    n_estimators = trial.suggest_int('n_estimators', 50, 300)
    max_depth = trial.suggest_int('max_depth', 5, 30)
    min_samples_split = trial.suggest_int('min_samples_split', 2, 20)
    min_samples_leaf = trial.suggest_int('min_samples_leaf', 1, 10)
    max_features = trial.suggest_categorical('max_features', ['sqrt', 'log2'])
    
    try:
        # Create pipeline with suggested hyperparameters
        pipeline = Pipeline([
            ("model", RandomForestClassifier(
                random_state=10,
                n_estimators=n_estimators,
                max_depth=max_depth,
                min_samples_split=min_samples_split,
                min_samples_leaf=min_samples_leaf,
                max_features=max_features,
                class_weight='balanced',
                n_jobs=-1
            ))
        ])
        
        # Use Stratified 5-Fold CV to evaluate
        cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=10)
        
        # Define scoring metrics
        scoring = {
            'accuracy': 'accuracy',
            'precision': 'precision_weighted',
            'recall': 'recall_weighted',
            'f1': 'f1_weighted',
            'roc_auc': 'roc_auc_ovr_weighted'
        }
        
        # Perform cross-validation
        cv_results = cross_validate(
            pipeline, 
            X_train, 
            y_train, 
            cv=cv, 
            scoring=scoring,
            return_train_score=False,
            n_jobs=1
        )
        
        # Return mean Accuracy for optimization
        mean_accuracy = cv_results['test_accuracy'].mean()
        
        return mean_accuracy
        
    except Exception as e:
        # If any error occurs, return a very poor score
        print(f"Trial failed with error: {str(e)[:50]}")
        return 0.0


# Create study object
study = optuna.create_study(
    direction='maximize',
    sampler=optuna.samplers.TPESampler(seed=10, n_startup_trials=20),
    pruner=optuna.pruners.PercentilePruner(percentile=25)
)

# Run optimization
print("Starting Optuna optimization for Random Forest...")
print("=" * 70)

study.optimize(
    objective, 
    n_trials=100, 
    show_progress_bar=True,
    gc_after_trial=True
)

# Get best trial
best_trial_rf = study.best_trial

print("\n" + "=" * 70)
print("OPTUNA TUNING RESULTS - RANDOM FOREST")
print("=" * 70)
print(f"\nBest Accuracy: {best_trial_rf.value:.4f}")
print(f"Number of Completed Trials: {len([t for t in study.trials if t.state.name == 'COMPLETE'])}")
print(f"\nBest Hyperparameters:")
print("-" * 70)
for key, value in best_trial_rf.params.items():
    if isinstance(value, float):
        print(f"  {key}: {value:.6f}")
    else:
        print(f"  {key}: {value}")

print("=" * 70)

# Store best params for next steps
best_params_rf = best_trial_rf.params

[I 2026-05-06 01:53:13,074] A new study created in memory with name: no-name-fbf7ed62-6ccc-4c18-bc47-2b1edd723436


Starting Optuna optimization for Random Forest...


  0%|          | 0/100 [00:00<?, ?it/s]

[I 2026-05-06 01:53:28,565] Trial 0 finished with value: 0.7237952788019777 and parameters: {'n_estimators': 243, 'max_depth': 5, 'min_samples_split': 14, 'min_samples_leaf': 8, 'max_features': 'sqrt'}. Best is trial 0 with value: 0.7237952788019777.
[I 2026-05-06 01:53:44,398] Trial 1 finished with value: 0.721614207194035 and parameters: {'n_estimators': 99, 'max_depth': 24, 'min_samples_split': 5, 'min_samples_leaf': 1, 'max_features': 'log2'}. Best is trial 0 with value: 0.7237952788019777.
[I 2026-05-06 01:53:51,237] Trial 2 finished with value: 0.7303566877110971 and parameters: {'n_estimators': 50, 'max_depth': 18, 'min_samples_split': 17, 'min_samples_leaf': 7, 'max_features': 'sqrt'}. Best is trial 2 with value: 0.7303566877110971.
[I 2026-05-06 01:54:30,411] Trial 3 finished with value: 0.7278090757397955 and parameters: {'n_estimators': 280, 'max_depth': 23, 'min_samples_split': 12, 'min_samples_leaf': 2, 'max_features': 'log2'}. Best is trial 2 with value: 0.730356687711097

# RFE Feature selection:

In [11]:
print("Starting RFE Feature Selection for Random Forest...")
print("=" * 80)
print(f"Using best hyperparameters from Optuna:")
for key, value in best_params_rf.items():
    if isinstance(value, float):
        print(f"  {key}: {value:.6f}")
    else:
        print(f"  {key}: {value}")
print("=" * 80)

# Feature count range to test (8-14 features)
feature_counts = range(8, 15)  # 8, 9, 10, 11, 12, 13, 14

# Dictionary to store results
rfe_results_rf = {}

# Stratified 5-Fold CV
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=10)

# Loop through each feature count
for n_features in feature_counts:
    print(f"\nTesting with {n_features} features...")
    
    # Store metrics for all folds
    fold_accuracies = []
    fold_precisions = []
    fold_recalls = []
    fold_f1s = []
    fold_roc_aucs = []
    
    # CV Loop - RFE EMBEDDED INSIDE
    fold_idx = 0
    for train_idx, val_idx in cv.split(X_train, y_train):
        fold_idx += 1
        
        # Split data into train and validation
        X_fold_train, X_fold_val = X_train.iloc[train_idx], X_train.iloc[val_idx]
        y_fold_train, y_fold_val = y_train.iloc[train_idx], y_train.iloc[val_idx]
        
        # Create pipeline with RFE
        pipeline = Pipeline([
            ("rfe", RFE(
                estimator=RandomForestClassifier(
                    random_state=10,
                    n_estimators=best_params_rf['n_estimators'],
                    max_depth=best_params_rf['max_depth'],
                    min_samples_split=best_params_rf['min_samples_split'],
                    min_samples_leaf=best_params_rf['min_samples_leaf'],
                    max_features=best_params_rf['max_features'],
                    class_weight='balanced',
                    n_jobs=-1
                ),
                n_features_to_select=n_features,
                step=1
            )),
            ("model", RandomForestClassifier(
                random_state=10,
                n_estimators=best_params_rf['n_estimators'],
                max_depth=best_params_rf['max_depth'],
                min_samples_split=best_params_rf['min_samples_split'],
                min_samples_leaf=best_params_rf['min_samples_leaf'],
                max_features=best_params_rf['max_features'],
                class_weight='balanced',
                n_jobs=-1
            ))
        ])
        
        # Fit on TRAINING fold (RFE learns feature importance from training fold only)
        pipeline.fit(X_fold_train, y_fold_train)
        
        # Predict on VALIDATION fold
        y_pred = pipeline.predict(X_fold_val)
        y_pred_proba = pipeline.predict_proba(X_fold_val)[:, 1]
        
        # Calculate metrics
        accuracy = accuracy_score(y_fold_val, y_pred)
        precision = precision_score(y_fold_val, y_pred, average='weighted')
        recall = recall_score(y_fold_val, y_pred, average='weighted')
        f1 = f1_score(y_fold_val, y_pred, average='weighted')
        roc_auc = roc_auc_score(y_fold_val, y_pred_proba, average='weighted')
        
        # Store results
        fold_accuracies.append(accuracy)
        fold_precisions.append(precision)
        fold_recalls.append(recall)
        fold_f1s.append(f1)
        fold_roc_aucs.append(roc_auc)
    
    # Store mean and std for this feature count
    rfe_results_rf[n_features] = {
        'accuracy': np.array(fold_accuracies),
        'precision': np.array(fold_precisions),
        'recall': np.array(fold_recalls),
        'f1': np.array(fold_f1s),
        'roc_auc': np.array(fold_roc_aucs)
    }
    
    # Display results for this feature count
    mean_accuracy = np.mean(fold_accuracies)
    std_accuracy = np.std(fold_accuracies)
    print(f"  Accuracy: {mean_accuracy:.4f} ± {std_accuracy:.4f}")

# ============================================================
# SELECT BEST FEATURE COUNT (based on Accuracy)
# ============================================================

print("\n" + "=" * 80)
print("RFE RESULTS SUMMARY (Random Forest)")
print("=" * 80)

# Create results dataframe
results_data = {}
for n_features in feature_counts:
    results_data[n_features] = {
        'Accuracy': f"{rfe_results_rf[n_features]['accuracy'].mean():.3f} ± {rfe_results_rf[n_features]['accuracy'].std():.3f}",
        'Precision': f"{rfe_results_rf[n_features]['precision'].mean():.3f} ± {rfe_results_rf[n_features]['precision'].std():.3f}",
        'Recall': f"{rfe_results_rf[n_features]['recall'].mean():.3f} ± {rfe_results_rf[n_features]['recall'].std():.3f}",
        'F1 Score': f"{rfe_results_rf[n_features]['f1'].mean():.3f} ± {rfe_results_rf[n_features]['f1'].std():.3f}",
        'ROC AUC': f"{rfe_results_rf[n_features]['roc_auc'].mean():.3f} ± {rfe_results_rf[n_features]['roc_auc'].std():.3f}"
    }

results_df = pd.DataFrame(results_data).T
print("\n")
print(results_df)

# Find best feature count based on Accuracy
best_feature_count_rf = max(
    rfe_results_rf.keys(),
    key=lambda x: rfe_results_rf[x]['accuracy'].mean()
)

best_accuracy = rfe_results_rf[best_feature_count_rf]['accuracy'].mean()
best_accuracy_std = rfe_results_rf[best_feature_count_rf]['accuracy'].std()

print("\n" + "=" * 80)
print("BEST FEATURE COUNT")
print("=" * 80)
print(f"\nBest Number of Features: {best_feature_count_rf}")
print(f"Best Accuracy: {best_accuracy:.3f} ± {best_accuracy_std:.3f}")
print("\nAll Metrics for Best Feature Count:")
print("-" * 80)
for metric_name in ['accuracy', 'precision', 'recall', 'f1', 'roc_auc']:
    metric_values = rfe_results_rf[best_feature_count_rf][metric_name]
    mean = metric_values.mean()
    std = metric_values.std()
    print(f"  {metric_name.upper():10s}: {mean:.3f} ± {std:.3f}")

print("=" * 80)

# Store best feature count for next step
best_rfe_features_rf = best_feature_count_rf
print(f"\n✓ Best feature count saved: {best_rfe_features_rf}")

Starting RFE Feature Selection for Random Forest...
Using best hyperparameters from Optuna:
  n_estimators: 169
  max_depth: 12
  min_samples_split: 3
  min_samples_leaf: 10
  max_features: log2

Testing with 8 features...
  Accuracy: 0.7276 ± 0.0050

Testing with 9 features...
  Accuracy: 0.7273 ± 0.0051

Testing with 10 features...
  Accuracy: 0.7298 ± 0.0046

Testing with 11 features...
  Accuracy: 0.7302 ± 0.0043

Testing with 12 features...
  Accuracy: 0.7311 ± 0.0045

Testing with 13 features...
  Accuracy: 0.7312 ± 0.0045

Testing with 14 features...
  Accuracy: 0.7320 ± 0.0047

RFE RESULTS SUMMARY (Random Forest)


         Accuracy      Precision         Recall       F1 Score        ROC AUC
8   0.728 ± 0.005  0.729 ± 0.005  0.728 ± 0.005  0.727 ± 0.005  0.793 ± 0.006
9   0.727 ± 0.005  0.728 ± 0.005  0.727 ± 0.005  0.727 ± 0.005  0.793 ± 0.006
10  0.730 ± 0.005  0.731 ± 0.005  0.730 ± 0.005  0.729 ± 0.005  0.795 ± 0.006
11  0.730 ± 0.004  0.731 ± 0.004  0.730 ± 0.004  0.730 ± 

# Final RF Pipeline with results

In [12]:
# Stratified 5-Fold CV
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=10)

# Store metrics for all folds
fold_accuracies = []
fold_precisions = []
fold_recalls = []
fold_f1s = []
fold_roc_aucs = []

print("\nRunning 5-Fold Stratified Cross-Validation...")
print("-" * 80)
print(f"{'Fold':<6} {'Accuracy':<12} {'Precision':<12} {'Recall':<12} {'F1 Score':<12} {'ROC AUC':<12}")
print("-" * 80)

# CV Loop - RFE EMBEDDED INSIDE
fold_idx = 0
for train_idx, val_idx in cv.split(X_train, y_train):
    fold_idx += 1
    
    # Split data into train and validation
    X_fold_train, X_fold_val = X_train.iloc[train_idx], X_train.iloc[val_idx]
    y_fold_train, y_fold_val = y_train.iloc[train_idx], y_train.iloc[val_idx]
    
    # Create pipeline with RFE and best feature count
    pipeline = Pipeline([
        ("rfe", RFE(
            estimator=RandomForestClassifier(
                random_state=10,
                n_estimators=best_params_rf['n_estimators'],
                max_depth=best_params_rf['max_depth'],
                min_samples_split=best_params_rf['min_samples_split'],
                min_samples_leaf=best_params_rf['min_samples_leaf'],
                max_features=best_params_rf['max_features'],
                class_weight='balanced',
                n_jobs=-1
            ),
            n_features_to_select=best_rfe_features_rf,
            step=1
        )),
        ("model", RandomForestClassifier(
            random_state=10,
            n_estimators=best_params_rf['n_estimators'],
            max_depth=best_params_rf['max_depth'],
            min_samples_split=best_params_rf['min_samples_split'],
            min_samples_leaf=best_params_rf['min_samples_leaf'],
            max_features=best_params_rf['max_features'],
            class_weight='balanced',
            n_jobs=-1
        ))
    ])
    
    # Fit on TRAINING fold
    pipeline.fit(X_fold_train, y_fold_train)
    
    # Predict on VALIDATION fold
    y_pred = pipeline.predict(X_fold_val)
    y_pred_proba = pipeline.predict_proba(X_fold_val)[:, 1]
    
    # Calculate metrics
    accuracy = accuracy_score(y_fold_val, y_pred)
    precision = precision_score(y_fold_val, y_pred, average='weighted')
    recall = recall_score(y_fold_val, y_pred, average='weighted')
    f1 = f1_score(y_fold_val, y_pred, average='weighted')
    roc_auc = roc_auc_score(y_fold_val, y_pred_proba, average='weighted')
    
    # Store results
    fold_accuracies.append(accuracy)
    fold_precisions.append(precision)
    fold_recalls.append(recall)
    fold_f1s.append(f1)
    fold_roc_aucs.append(roc_auc)
    
    # Print fold results
    print(f"{fold_idx:<6} {accuracy:<12.4f} {precision:<12.4f} {recall:<12.4f} {f1:<12.4f} {roc_auc:<12.4f}")

# Calculate and display summary
print("\n" + "=" * 80)
print("FINAL PIPELINE RESULTS (Mean ± Std)")
print("=" * 80)

print("\n")
print(f"{'Accuracy':<12s}: {np.mean(fold_accuracies):.3f} ± {np.std(fold_accuracies):.3f}")
print(f"{'Precision':<12s}: {np.mean(fold_precisions):.3f} ± {np.std(fold_precisions):.3f}")
print(f"{'Recall':<12s}: {np.mean(fold_recalls):.3f} ± {np.std(fold_recalls):.3f}")
print(f"{'F1 Score':<12s}: {np.mean(fold_f1s):.3f} ± {np.std(fold_f1s):.3f}")
print(f"{'ROC AUC':<12s}: {np.mean(fold_roc_aucs):.3f} ± {np.std(fold_roc_aucs):.3f}")

print("\n" + "=" * 80)

# ============================================================
# TRAIN FINAL MODEL ON FULL TRAIN DATA
# ============================================================

final_pipeline_rf = Pipeline([
    ("rfe", RFE(
        estimator=RandomForestClassifier(
            random_state=10,
            n_estimators=best_params_rf['n_estimators'],
            max_depth=best_params_rf['max_depth'],
            min_samples_split=best_params_rf['min_samples_split'],
            min_samples_leaf=best_params_rf['min_samples_leaf'],
            max_features=best_params_rf['max_features'],
            class_weight='balanced',
            n_jobs=-1
        ),
        n_features_to_select=best_rfe_features_rf,
        step=1
    )),
    ("model", RandomForestClassifier(
        random_state=10,
        n_estimators=best_params_rf['n_estimators'],
        max_depth=best_params_rf['max_depth'],
        min_samples_split=best_params_rf['min_samples_split'],
        min_samples_leaf=best_params_rf['min_samples_leaf'],
        max_features=best_params_rf['max_features'],
        class_weight='balanced',
        n_jobs=-1
    ))
])

print("Training Final Model on full train data...")
final_pipeline_rf.fit(X_train, y_train)
print("✓ Final model trained on full training data!")
print("=" * 80)



Running 5-Fold Stratified Cross-Validation...
--------------------------------------------------------------------------------
Fold   Accuracy     Precision    Recall       F1 Score     ROC AUC     
--------------------------------------------------------------------------------
1      0.7400       0.7414       0.7400       0.7394       0.8082      
2      0.7293       0.7307       0.7293       0.7286       0.7922      
3      0.7335       0.7347       0.7335       0.7329       0.7987      
4      0.7260       0.7273       0.7260       0.7253       0.7928      
5      0.7312       0.7331       0.7312       0.7303       0.7948      

FINAL PIPELINE RESULTS (Mean ± Std)


Accuracy    : 0.732 ± 0.005
Precision   : 0.733 ± 0.005
Recall      : 0.732 ± 0.005
F1 Score    : 0.731 ± 0.005
ROC AUC     : 0.797 ± 0.006

Training Final Model on full train data...
✓ Final model trained on full training data!
